# AI Intervention Example
Demonstrates task-adaptive AI support based on detected user activity.

## Install Required Libraries
Run the next cell once to install the dependencies for this notebook.

In [ ]:
# Install required libraries
%pip install openai ipython flask

## 1. Import Required Libraries
Load the OpenAI client, display helpers, image encoding utilities, and Flask for the optional service endpoint.

In [ ]:

import base64
from pathlib import Path
from openai import OpenAI
from IPython.display import display, Image
from flask import Flask, request, jsonify
import threading
import os


## 2. Select the Detected Activity
For the tutorial, choose one activity label to simulate the activity detected from gaze features.

In [ ]:
# Detected user activity — change to 'Inspection', 'Search', 'Reading'
detected_task = "Search"

## 3. Configure the API Key
Set the OpenRouter API key used by the OpenAI-compatible client.

In [ ]:
OPENROUTER_API_KEY = ## PUT YOUR OPENROUTER API KEY HERE ## 
#You can get an API key from https://openrouter.ai/. We will share one during the tutorial.

## 4. Configure the Client, Demo Images, and Prompts
The next cell sets up the API client, locates the demo images, and maps each detected task to an intervention prompt.

In [ ]:
# Set up openAI client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Specify the folder with the demo images.
DEMO_IMAGES ="DemoImages"

# Define prompts for the different user activities
PROMPTS = {
    "Reading":    "Summarize the text visible in this image into concise bullet points.",
    "Inspection": "Describe the robot shown in this image and highlight its key features.",
    "Search":     "Locate the green pin in this image and describe where it is positioned.",
}


## 5. Define the Intervention Function
This helper loads the activity-specific image, sends it with the matching prompt, and returns the generated support text.

In [ ]:
def image_path_for_task(detected_task):
    if detected_task not in PROMPTS:
        valid_tasks = ", ".join(PROMPTS.keys())
        raise ValueError(f"Unknown detected_task {detected_task!r}. Use one of: {valid_tasks}")

    image_path = str(DEMO_IMAGES) + f"/{detected_task}_example.png"
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Demo image not found: {image_path}")
    return image_path


def run_task_for_image(detected_task):
    image_path = image_path_for_task(detected_task)
    with open(image_path, "rb") as f:
        b64_image = base64.b64encode(f.read()).decode()

    response = client.chat.completions.create(
        model="openai/gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_image}"}},
                    {"type": "text",      "text": PROMPTS[detected_task]},
                ],
            }
        ],
    )
    return str(response.choices[0].message.content), image_path


## 6. Available Demo Tasks
These are the image files used for each simulated activity.

| Task | Image |
|---|---|
| `reading` | `demo_images/reading_example.png` |
| `inspection` | `demo_images/inspection_example.png` |
| `search` | `demo_images/search_example.png` |

## 7. Run the Intervention Once
Display the selected image and print the AI response for the current `detected_task`.

In [ ]:
# Load the demo image associated with the user activity
image_path = image_path_for_task(detected_task)
display(Image(filename=str(image_path)))

response_text, _ = run_task_for_image(detected_task)
print(f"Task: {detected_task}\n")
print(response_text)


## Expose as a service
This section starts a lightweight HTTP service that accepts a GET parameter named `detected_task`.
You can still run the notebook normally for testing, and then execute the service cell to expose the same function as a reusable endpoint.


In [ ]:
app = Flask(__name__)

@app.route("/intervention")
def intervention():
    predicted_activity = request.args.get("predicted_activity")
    if not predicted_activity:
        return jsonify(error="Missing predicted_activity parameter"), 400
    if predicted_activity not in PROMPTS:
        return jsonify(
            error="Invalid predicted_activity",
            valid_tasks=list(PROMPTS.keys()),
        ), 400

    response_text, _ = run_task_for_image(predicted_activity)
    return jsonify(task=predicted_activity, response=response_text)

def start_service():
    thread = threading.Thread(
        target=app.run,
        kwargs={
            "host": "0.0.0.0",
            "port": 5020,
            "debug": False,
            "use_reloader": False,
        },
        daemon=True,
    )
    thread.start()
    print("Service running at http://127.0.0.1:5020/intervention")
    return thread

service_thread = start_service()
